# Phase 1 — Lesson 14: Norms and Distances
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 14: Norms and Distances
**Type:** Build | **Languages:** Python | **Time:** ~90 minutes
**Prerequisites:** Phase 1, Lessons 01, 02



# What You'll Learn

---

## Learning Objectives

- Implement L1, L2, cosine, Mahalanobis, Jaccard, and edit distance functions from scratch
- Select the appropriate distance metric for a given ML task and explain why alternatives fail
- Connect L1 and L2 norms to LASSO and Ridge regularization and their geometric constraint regions


# Distance Defines Similarity

---

Every search, clustering, or recommendation system answers: how similar are these two items? The distance function you choose determines the answer.


In [ ]:
import numpy as npimport mathx = np.array([1, 2, 3])y = np.array([4, 5, 6])l1 = np.sum(np.abs(x - y))l2 = np.sqrt(np.sum((x - y) ** 2))cos_sim = np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))print(f"L1 distance:   {l1}")print(f"L2 distance:   {l2:.4f}")print(f"Cosine sim:    {cos_sim:.4f}")print(f"Cosine dist:   {1 - cos_sim:.4f}")# Cosine similarity is scale-invariantprint(f"\ncosine(2x, y) = {np.dot(2*x, y) / (np.linalg.norm(2*x) * np.linalg.norm(y)):.4f}")print(f"cosine(x, y)  = {cos_sim:.4f}")print("Cosine similarity is invariant to scaling.")

In [ ]:
import matplotlib.pyplot as pltimport numpy as np# Unit balls for different Lp normstheta = np.linspace(0, 2*np.pi, 200)fig, axes = plt.subplots(1, 3, figsize=(12, 4))for ax, p, label in [(axes[0], 1, 'L1 (diamond)'), (axes[1], 2, 'L2 (circle)'), (axes[2], np.inf, 'L-inf (square)')]:    if p == np.inf:        xs = np.clip(np.cos(theta), -1, 1)        ys = np.clip(np.sin(theta), -1, 1)    else:        xs = np.sign(np.cos(theta)) * np.abs(np.cos(theta)) ** (1/p)        ys = np.sign(np.sin(theta)) * np.abs(np.sin(theta)) ** (1/p)    ax.plot(xs, ys, lw=2, color='#e74c3c')    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)    ax.set_aspect('equal'); ax.set_title(label, fontweight='bold')    ax.grid(True, alpha=0.3)plt.suptitle('Unit Balls for Different Lp Norms', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

In [ ]:
import numpy as np# Mahalanobis distance for correlated featuresdef mahalanobis(x, y, cov_inv):    d = x - y    return np.sqrt(d @ cov_inv @ d)cov = np.array([[1.0, 0.9], [0.9, 1.0]])cov_inv = np.linalg.inv(cov)p1 = np.array([2.0, 2.0])p2 = np.array([2.1, 2.1])p3 = np.array([2.0, 2.5])print(f"Euclidean p1-p2: {np.linalg.norm(p1-p2):.4f}")print(f"Mahalanobis p1-p2: {mahalanobis(p1, p2, cov_inv):.4f}")print(f"Euclidean p1-p3: {np.linalg.norm(p1-p3):.4f}")print(f"Mahalanobis p1-p3: {mahalanobis(p1, p3, cov_inv):.4f}")print("\nMahalanobis correctly downweights correlated direction.")

In [ ]:
import numpy as npdef edit_distance(s1, s2):    m, n = len(s1), len(s2)    dp = [[0] * (n + 1) for _ in range(m + 1)]    for i in range(m + 1): dp[i][0] = i    for j in range(n + 1): dp[0][j] = j    for i in range(1, m + 1):        for j in range(1, n + 1):            cost = 0 if s1[i-1] == s2[j-1] else 1            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)    return dp[m][n]print(f"edit_distance('kitten', 'sitting') = {edit_distance('kitten', 'sitting')}")print(f"edit_distance('flaw', 'lawn') = {edit_distance('flaw', 'lawn')}")

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Same data, different distances → different nearest neighborsnp.random.seed(42)points = np.random.randn(10, 2)query = np.array([0.5, 0.5])dists_l1 = np.sum(np.abs(points - query), axis=1)dists_l2 = np.sqrt(np.sum((points - query) ** 2, axis=1))nn_l1 = np.argmin(dists_l1)nn_l2 = np.argmin(dists_l2)fig, ax = plt.subplots(figsize=(6, 6))ax.scatter(points[:, 0], points[:, 1], c='gray', s=50, alpha=0.6)ax.scatter(query[0], query[1], c='red', s=100, marker='*', label='Query')ax.scatter(points[nn_l1, 0], points[nn_l1, 1], c='blue', s=100, marker='s', label=f'NN L1 ({nn_l1})')ax.scatter(points[nn_l2, 0], points[nn_l2, 1], c='green', s=100, marker='^', label=f'NN L2 ({nn_l2})')ax.set_title('Same Data, Different Distances → Different Neighbors', fontsize=12, fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()print(f"L1 nearest neighbor: point {nn_l1}")print(f"L2 nearest neighbor: point {nn_l2}")

# L1 vs L2 Regularization

---

| Property | L1 (Lasso) | L2 (Ridge) |
|----------|------------|------------|
| Penalty | sum of |w| | sum of w² |
| Effect | Sparse weights (some exactly 0) | All weights shrink toward 0 |
| Constraint shape | Diamond | Circle |
| Use case | Feature selection | Weight shrinkage |

L1 produces sparsity because the diamond has corners on the axes — loss contours touch corners where one weight is zero.


# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** When is cosine similarity better than L2?
> **Explanation:** When magnitude is noise and direction matters — text embeddings, RAG.

**Q2:** When is L1 better than L2?
> **Explanation:** When outliers are present — L1 doesn't square large errors.

**Q3:** What does Mahalanobis distance account for?
> **Explanation:** Feature correlations and different scales via the covariance matrix.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Create two vectors with high cosine similarity but large L2 distance (same direction, different magnitude).

**Exercise 2:** Implement a k-nearest neighbors classifier using your distance functions.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| L1 norm | Manhattan distance | Sum of absolute values — robust to outliers |
| L2 norm | Euclidean distance | Straight-line distance — most common in ML |
| Cosine similarity | Angle between vectors | Magnitude-invariant — used in embedding search |
| Mahalanobis distance | Correlation-aware distance | L2 in whitened space using inverse covariance |
| Jaccard distance | Set overlap | 1 - |intersection|/|union| — for sets, not vectors |
| Edit distance | Levenshtein distance | Minimum edits to transform one string into another |


# Further Reading

---

- [FAISS: Efficient Similarity Search](https://github.com/facebookresearch/faiss)
- [scikit-learn neighbors](https://scikit-learn.org/stable/modules/neighbors.html)


# Next Steps

---

When you're ready, continue to Lesson 15: Statistics for Machine Learning.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
